> Repository edition: run from the repository root using an environment with the dependencies in `requirements.txt`. Machine-specific environment paths have been replaced. Historical outputs are retained except environment checks; they are not evidence of a fresh rerun. See `docs/REPRODUCIBILITY.md`.


# CP_22 — Round 11
## Black-box optimisation: examining recurring regions and distances

This notebook prepares one new query for each of the eight black-box
functions using the initial observations and feedback from ten completed
rounds.

The workflow follows the previous notebooks:

1. Check the environment and validate the cumulative data.
2. Review improvements and prediction errors from round 10.
3. Examine recurring regions, distances and sampling gaps within each function.
4. Fit the surrogate models and compare candidate queries.
5. Select and validate eight queries for round 11.

The clustering analysis will help assess whether frequently sampled
regions also contain consistently strong observations. Any groupings
will be interpreted cautiously because the observations were collected
adaptively and cover only a small part of the search space.

Proposed queries and model predictions will remain separate from observed
function evaluations. Round-11 outcomes are not yet available.

In [ ]:
# CP_22 — Round 11
# Check the Python environment and locate the project data.

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

project_path = Path.cwd()

# Use the selected notebook kernel; no machine-specific path is required.
expected_python = Path(sys.executable)

initial_data_path = project_path / "initial_data"
feedback_path = project_path / "week_10"

inputs_path = feedback_path / "inputs.txt"
outputs_path = feedback_path / "outputs.txt"

# Expected round numbers; file contents will be validated next.
completed_rounds = 10
submission_round = 11

correct_kernel = (
    Path(sys.executable).resolve() == expected_python.resolve()
)

print("Python executable:", sys.executable)
print("Working directory:", Path.cwd())
print("Project folder:", project_path)

print("\nSelected interpreter check:", correct_kernel)
print("Project folder found:", project_path.is_dir())
print("Initial data folder found:", initial_data_path.is_dir())
print("Feedback folder found:", feedback_path.is_dir())
print("Cumulative inputs found:", inputs_path.is_file())
print("Cumulative outputs found:", outputs_path.is_file())

print("\nExpected completed rounds:", completed_rounds)
print("Round to prepare:", submission_round)

assert correct_kernel, (
    "Select the notebook kernel corresponding to: "
    f"{expected_python}"
)
assert project_path.is_dir(), (
    f"Project folder not found: {project_path}"
)
assert initial_data_path.is_dir(), (
    f"Initial data folder not found: {initial_data_path}"
)
assert feedback_path.is_dir(), (
    f"Feedback folder not found: {feedback_path}"
)
assert inputs_path.is_file(), (
    f"Inputs file not found: {inputs_path}"
)
assert outputs_path.is_file(), (
    f"Outputs file not found: {outputs_path}"
)

print("\nEnvironment and file checks passed.")
print(
    f"Ready to load and validate the {completed_rounds} feedback rounds."
)

## 2. Load and validate the cumulative observations

The initial observations are combined with ten completed feedback rounds
for each function. Each round must contain one query and one observed
output for each of the eight functions.

Validation checks the number of rounds, input dimensions, finite numerical
values and input bounds. Negative and extremely small outputs are retained.

The cumulative feedback files are loaded once; earlier weekly snapshots
are not appended again, which avoids counting the same evaluations twice.

In [2]:
# Load the initial observations and ten cumulative feedback rounds.

import ast
import numpy as np

required_variables = [
    "initial_data_path",
    "inputs_path",
    "outputs_path",
    "completed_rounds",
    "submission_round",
]

missing_variables = [
    name for name in required_variables if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Run the first code cell before this cell. "
        f"Missing variables: {', '.join(missing_variables)}"
    )


class ArrayLiteralTransformer(ast.NodeTransformer):
    """Convert array([...]) wrappers into literal lists without executing code."""

    def visit_Call(self, node):
        is_array = (
            isinstance(node.func, ast.Name)
            and node.func.id == "array"
        )
        is_numpy_array = (
            isinstance(node.func, ast.Attribute)
            and isinstance(node.func.value, ast.Name)
            and node.func.value.id in {"np", "numpy"}
            and node.func.attr == "array"
        )

        if not (is_array or is_numpy_array):
            raise ValueError("Unsupported function call in feedback file.")

        if len(node.args) != 1 or node.keywords:
            raise ValueError(
                "Expected an array wrapper with one argument and no keywords."
            )

        return self.visit(node.args[0])


def read_feedback_rounds(path):
    """Read successive literal round lists safely, without eval()."""
    text = path.read_text(encoding="utf-8-sig").strip()

    if not text:
        raise ValueError(f"Feedback file is empty: {path}")

    tree = ast.parse(text, mode="exec")
    rounds = []

    for statement in tree.body:
        if not isinstance(statement, ast.Expr):
            raise ValueError(
                f"Expected literal round lists only in: {path.name}"
            )

        literal = ArrayLiteralTransformer().visit(statement.value)
        round_values = ast.literal_eval(literal)

        if not isinstance(round_values, (list, tuple)):
            raise ValueError(
                f"Each round must be a list or tuple in: {path.name}"
            )

        rounds.append(round_values)

    return rounds


function_dimensions = {
    1: 2,
    2: 2,
    3: 3,
    4: 4,
    5: 4,
    6: 5,
    7: 6,
    8: 8,
}

expected_initial_counts = {
    1: 10,
    2: 10,
    3: 15,
    4: 30,
    5: 20,
    6: 20,
    7: 30,
    8: 40,
}

input_rounds = read_feedback_rounds(inputs_path)
output_rounds = read_feedback_rounds(outputs_path)

assert len(input_rounds) == completed_rounds, (
    f"Expected {completed_rounds} input rounds; "
    f"found {len(input_rounds)}."
)
assert len(output_rounds) == completed_rounds, (
    f"Expected {completed_rounds} output rounds; "
    f"found {len(output_rounds)}."
)

for round_number, (round_inputs, round_outputs) in enumerate(
    zip(input_rounds, output_rounds), start=1
):
    assert len(round_inputs) == 8, (
        f"Round {round_number}: expected inputs for eight functions."
    )
    assert len(round_outputs) == 8, (
        f"Round {round_number}: expected outputs for eight functions."
    )

initial_data = {}
feedback_data = {}
updated_data = {}

for function_id in range(1, 9):
    dimensions = function_dimensions[function_id]
    initial_count = expected_initial_counts[function_id]

    function_path = initial_data_path / f"function_{function_id}"
    initial_inputs_file = function_path / "initial_inputs.npy"
    initial_outputs_file = function_path / "initial_outputs.npy"

    assert initial_inputs_file.is_file(), (
        f"Initial inputs not found: {initial_inputs_file}"
    )
    assert initial_outputs_file.is_file(), (
        f"Initial outputs not found: {initial_outputs_file}"
    )

    X_initial = np.asarray(
        np.load(initial_inputs_file, allow_pickle=False),
        dtype=float,
    )
    y_initial = np.asarray(
        np.load(initial_outputs_file, allow_pickle=False),
        dtype=float,
    ).reshape(-1)

    assert X_initial.shape == (initial_count, dimensions), (
        f"Function {function_id}: unexpected initial input shape "
        f"{X_initial.shape}."
    )
    assert y_initial.shape == (initial_count,), (
        f"Function {function_id}: unexpected initial output shape "
        f"{y_initial.shape}."
    )

    feedback_inputs = []
    feedback_outputs = []

    for round_index in range(completed_rounds):
        query = np.asarray(
            input_rounds[round_index][function_id - 1],
            dtype=float,
        )
        output = np.asarray(
            output_rounds[round_index][function_id - 1],
            dtype=float,
        )

        assert query.shape == (dimensions,), (
            f"Function {function_id}, round {round_index + 1}: "
            f"expected {dimensions} coordinates; found shape {query.shape}."
        )
        assert output.size == 1, (
            f"Function {function_id}, round {round_index + 1}: "
            "expected one scalar output."
        )

        feedback_inputs.append(query)
        feedback_outputs.append(output.item())

    X_feedback = np.asarray(feedback_inputs, dtype=float)
    y_feedback = np.asarray(feedback_outputs, dtype=float)

    X = np.vstack([X_initial, X_feedback])
    y = np.concatenate([y_initial, y_feedback])

    assert np.isfinite(X).all(), (
        f"Function {function_id}: non-finite input values."
    )
    assert np.isfinite(y).all(), (
        f"Function {function_id}: non-finite output values."
    )
    assert ((X >= 0.0) & (X <= 1.0)).all(), (
        f"Function {function_id}: input coordinates outside [0, 1]."
    )
    assert X.shape == (initial_count + completed_rounds, dimensions)
    assert y.shape == (initial_count + completed_rounds,)

    initial_data[function_id] = {
        "X": X_initial,
        "y": y_initial,
    }
    feedback_data[function_id] = {
        "X": X_feedback,
        "y": y_feedback,
    }
    updated_data[function_id] = {
        "X": X,
        "y": y,
    }

    print(
        f"Function {function_id}: "
        f"{initial_count} → {len(y)} observations | "
        f"{dimensions} variables"
    )

total_observations = sum(
    len(data["y"]) for data in updated_data.values()
)

print("\nValidated feedback rounds:", len(input_rounds))
print("Round to prepare:", submission_round)
print("Total evaluated observations:", total_observations)
print("All eight datasets loaded and validated successfully.")

Function 1: 10 → 20 observations | 2 variables
Function 2: 10 → 20 observations | 2 variables
Function 3: 15 → 25 observations | 3 variables
Function 4: 30 → 40 observations | 4 variables
Function 5: 20 → 30 observations | 4 variables
Function 6: 20 → 30 observations | 5 variables
Function 7: 30 → 40 observations | 6 variables
Function 8: 40 → 50 observations | 8 variables

Validated feedback rounds: 10
Round to prepare: 11
Total evaluated observations: 255
All eight datasets loaded and validated successfully.


## 3. Review observed improvements through round 10

For each function, the latest observed output is compared with the best
value available before round 10. Because the objective is maximisation,
an improvement occurs when the new output is strictly higher than the
previous best, including when both values are negative.

The historical summary records which functions improved in each round.
Improvement magnitudes are interpreted separately for each function
because their output scales differ substantially.

These results describe observed progress; they do not establish that a
global optimum has been reached.

In [3]:
# Review the latest round and reconstruct the improvement history.

import numpy as np
import pandas as pd
from IPython.display import display

required_variables = [
    "initial_data",
    "feedback_data",
    "updated_data",
    "completed_rounds",
]

missing_variables = [
    name for name in required_variables if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Run the preceding data-loading cell first. "
        f"Missing variables: {', '.join(missing_variables)}"
    )

latest_round_rows = []
history_rows = []

for function_id in range(1, 9):
    y_initial = initial_data[function_id]["y"]
    y_feedback = feedback_data[function_id]["y"]

    assert len(y_feedback) == completed_rounds, (
        f"Function {function_id}: unexpected feedback count."
    )

    best_so_far = float(np.max(y_initial))

    for round_number, observed_output in enumerate(y_feedback, start=1):
        observed_output = float(observed_output)
        previous_best = best_so_far
        improved = observed_output > previous_best
        improvement = max(0.0, observed_output - previous_best)
        best_so_far = max(previous_best, observed_output)

        history_rows.append({
            "Function": function_id,
            "Round": round_number,
            "Best before round": previous_best,
            "Observed output": observed_output,
            "Improved": improved,
            "Improvement": improvement,
            "Best so far": best_so_far,
        })

        if round_number == completed_rounds:
            latest_round_rows.append({
                "Function": function_id,
                f"Best before round {completed_rounds}": previous_best,
                f"Round {completed_rounds} output": observed_output,
                f"Round {completed_rounds} improved": improved,
                "Improvement": improvement,
                "Current best": best_so_far,
            })

    assert best_so_far == float(
        np.max(updated_data[function_id]["y"])
    ), f"Function {function_id}: inconsistent best-so-far value."

improvement_history_df = pd.DataFrame(history_rows)

round_comparison_df = (
    pd.DataFrame(latest_round_rows)
    .set_index("Function")
    .sort_index()
)

numeric_columns = [
    f"Best before round {completed_rounds}",
    f"Round {completed_rounds} output",
    "Improvement",
    "Current best",
]

display(
    round_comparison_df.style.format({
        column: "{:.6e}" for column in numeric_columns
    })
)

for round_number in range(1, completed_rounds + 1):
    improved_functions = (
        improvement_history_df.loc[
            (improvement_history_df["Round"] == round_number)
            & improvement_history_df["Improved"],
            "Function",
        ]
        .sort_values()
        .tolist()
    )

    print(
        f"Functions improved in round {round_number}: "
        f"{improved_functions} | Count: {len(improved_functions)}"
    )

print(
    "\nImprovements are calculated from observed outputs, "
    "not model predictions."
)

,Best before round 10,Round 10 output,Round 10 improved,Improvement,Current best
Function,,,,,
1,7.710875e-16,-5.418764e-183,False,0.000000e+00,7.710875e-16
2,6.582395e-01,4.565338e-01,False,0.000000e+00,6.582395e-01
3,-3.096391e-02,-5.532620e-02,False,0.000000e+00,-3.096391e-02
4,6.152729e-01,5.801313e-01,False,0.000000e+00,6.152729e-01
5,8.510289e+03,8.515379e+03,True,5.089256e+00,8.515379e+03
6,-2.424310e-01,-3.355773e-01,False,0.000000e+00,-2.424310e-01
7,2.374721e+00,2.436323e+00,True,6.160196e-02,2.436323e+00
8,9.958227e+00,9.902243e+00,False,0.000000e+00,9.958227e+00


Functions improved in round 1: [2, 4, 5, 7, 8] | Count: 5
Functions improved in round 2: [6, 8] | Count: 2
Functions improved in round 3: [5, 8] | Count: 2
Functions improved in round 4: [6, 8] | Count: 2
Functions improved in round 5: [3, 4, 5, 8] | Count: 4
Functions improved in round 6: [5, 6, 7] | Count: 3
Functions improved in round 7: [5] | Count: 1
Functions improved in round 8: [4, 5, 7] | Count: 3
Functions improved in round 9: [8] | Count: 1
Functions improved in round 10: [5, 7] | Count: 2

Improvements are calculated from observed outputs, not model predictions.


## 4. Compare round-10 predictions with observed outputs

The predictions recorded in CP_21 are compared with the actual round-10
evaluations. Historical query coordinates are checked against the feedback
inputs before calculating prediction errors.

The comparison includes signed errors, absolute errors and absolute errors
divided by the GP predictive standard deviation. Auxiliary predictions are
also evaluated for Functions 5 and 7.

Historical predictions are taken from the displayed CP_21 table and are
therefore rounded. These individual error ratios are diagnostic indicators,
not a complete assessment of uncertainty calibration.

Round 10 improved the best observed outputs for Functions 5 and 7.
Function 5's small improvement at a near-repeat point motivates reviewing
the information gained from tightly concentrated local sampling.

In [4]:
# Compare recorded CP_21 predictions with round-10 feedback.
# Historical predictions below use the rounded values displayed in CP_21.

import numpy as np
import pandas as pd
from IPython.display import display

required_variables = ["feedback_data", "completed_rounds"]

missing_variables = [
    name for name in required_variables if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Run the preceding data-loading cell first. "
        f"Missing variables: {', '.join(missing_variables)}"
    )

historical_round = 10

assert completed_rounds == historical_round, (
    "This comparison is specifically for round 10."
)

historical_predictions = {
    1: {
        "query": "0.979455-0.999881",
        "mean": -1.009432e-04,
        "std": 7.547547e-04,
        "auxiliary": np.nan,
    },
    2: {
        "query": "0.713020-0.000033",
        "mean": 5.872322e-01,
        "std": 6.795209e-02,
        "auxiliary": np.nan,
    },
    3: {
        "query": "0.317762-0.998895-0.417393",
        "mean": -3.501582e-02,
        "std": 4.632375e-02,
        "auxiliary": np.nan,
    },
    4: {
        "query": "0.409023-0.410500-0.383078-0.419599",
        "mean": 3.606262e-01,
        "std": 2.124775e-01,
        "auxiliary": np.nan,
    },
    5: {
        "query": "0.997181-0.999005-0.999356-0.996804",
        "mean": 8.045722e+03,
        "std": 3.267279e+02,
        "auxiliary": 8.054748e+03,
    },
    6: {
        "query": "0.459114-0.364921-0.534349-0.679924-0.000139",
        "mean": -3.720083e-01,
        "std": 1.126321e-01,
        "auxiliary": np.nan,
    },
    7: {
        "query": (
            "0.021409-0.225288-0.479072-"
            "0.145922-0.286321-0.697760"
        ),
        "mean": 2.280249e+00,
        "std": 1.273900e-01,
        "auxiliary": 1.092263e+00,
    },
    8: {
        "query": (
            "0.071062-0.278596-0.059377-0.031881-"
            "0.988708-0.628793-0.187893-0.999301"
        ),
        "mean": 9.944405e+00,
        "std": 2.760331e-02,
        "auxiliary": np.nan,
    },
}

prediction_error_rows = []

for function_id in range(1, 9):
    record = historical_predictions[function_id]

    historical_query = np.array(
        [float(value) for value in record["query"].split("-")],
        dtype=float,
    )

    actual_query = feedback_data[function_id]["X"][
        historical_round - 1
    ]
    observed_output = float(
        feedback_data[function_id]["y"][historical_round - 1]
    )

    assert historical_query.shape == actual_query.shape, (
        f"Function {function_id}: historical query dimension mismatch."
    )
    assert np.allclose(
        historical_query,
        actual_query,
        rtol=0.0,
        atol=1e-8,
    ), (
        f"Function {function_id}: round-10 feedback inputs do not "
        "match the recorded CP_21 query. Check the files before continuing."
    )

    predicted_mean = float(record["mean"])
    predicted_std = float(record["std"])
    auxiliary_prediction = float(record["auxiliary"])

    assert np.isfinite(predicted_mean)
    assert np.isfinite(predicted_std) and predicted_std > 0

    signed_error = observed_output - predicted_mean
    absolute_error = abs(signed_error)

    auxiliary_error = (
        abs(observed_output - auxiliary_prediction)
        if np.isfinite(auxiliary_prediction)
        else np.nan
    )

    prediction_error_rows.append({
        "Function": function_id,
        "Predicted mean": predicted_mean,
        "Predicted std": predicted_std,
        "Observed output": observed_output,
        "Observed minus predicted": signed_error,
        "Absolute error / predicted std": (
            absolute_error / predicted_std
        ),
        "GP absolute error": absolute_error,
        "Auxiliary prediction": auxiliary_prediction,
        "Auxiliary absolute error": auxiliary_error,
    })

prediction_errors_df = (
    pd.DataFrame(prediction_error_rows)
    .set_index("Function")
    .sort_index()
)

error_formats = {
    column: "{:.6e}"
    for column in prediction_errors_df.columns
}
error_formats["Absolute error / predicted std"] = "{:.2f}"

display(
    prediction_errors_df.style.format(
        error_formats,
        na_rep="—",
    )
)

print(
    "\nAll eight historical queries match the round 10 feedback inputs."
)
print("Errors use rounded historical predictions from CP_21.")
print(
    "Error / predicted std is a diagnostic for these individual "
    "observations, not a complete uncertainty calibration assessment."
)

,Predicted mean,Predicted std,Observed output,Observed minus predicted,Absolute error / predicted std,GP absolute error,Auxiliary prediction,Auxiliary absolute error
Function,,,,,,,,
1,-1.009432e-04,7.547547e-04,-5.418764e-183,1.009432e-04,0.13,1.009432e-04,—,—
2,5.872322e-01,6.795209e-02,4.565338e-01,-1.306984e-01,1.92,1.306984e-01,—,—
3,-3.501582e-02,4.632375e-02,-5.532620e-02,-2.031038e-02,0.44,2.031038e-02,—,—
4,3.606262e-01,2.124775e-01,5.801313e-01,2.195051e-01,1.03,2.195051e-01,—,—
5,8.045722e+03,3.267279e+02,8.515379e+03,4.696566e+02,1.44,4.696566e+02,8.054748e+03,4.606306e+02
6,-3.720083e-01,1.126321e-01,-3.355773e-01,3.643103e-02,0.32,3.643103e-02,—,—
7,2.280249e+00,1.273900e-01,2.436323e+00,1.560743e-01,1.23,1.560743e-01,1.092263e+00,1.344060e+00
8,9.944405e+00,2.760331e-02,9.902243e+00,-4.216195e-02,1.53,4.216195e-02,—,—



All eight historical queries match the round 10 feedback inputs.
Errors use rounded historical predictions from CP_21.
Error / predicted std is a diagnostic for these individual observations, not a complete uncertainty calibration assessment.


## 5. Examine promising regions and sampling concentration

Each function is analysed separately because its inputs and outputs define
a different optimisation problem.

For each function, I examine the highest-ranked 25% of observed outputs
and compare their spatial concentration with that of the complete dataset.
This subset is defined by output rank; it is not an automatically
discovered cluster.

Distances are calculated in the original [0, 1] input coordinates.
Euclidean distance divided by the square root of dimensionality is used
as an RMS coordinate distance. This gives every coordinate equal weight,
an assumption that may overlook differences in variable relevance.

The analysis records:

- The coordinates of the best observed point.
- The centroid of the highest-ranked subset.
- Pairwise distances within that subset and within the complete dataset.
- The number and observed performance of neighbours around the best point.
- The distance from the latest query to earlier observations.

A neighbourhood radius of 0.10 RMS coordinate units is used as a
descriptive threshold. Counts at 0.05 and 0.15 are also reported to show
sensitivity to this choice. These thresholds do not establish natural
cluster boundaries.

A dense region is not necessarily a high-performing region: density can
reflect the sampling policy. Similarly, a centroid can fall between
separate promising regions and is not automatically a suitable query.

In [6]:
# Describe promising regions and sampling concentration.
# This diagnostic does not fit a clustering model or select new queries.

import numpy as np
import pandas as pd
from IPython.display import display

required_variables = [
    "updated_data",
    "initial_data",
    "completed_rounds",
]

missing_variables = [
    name for name in required_variables if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Run the preceding data-loading cell first. "
        f"Missing variables: {', '.join(missing_variables)}"
    )


def rms_distance_matrix(X):
    """Return Euclidean distances divided by sqrt(number of variables)."""
    differences = X[:, None, :] - X[None, :, :]
    return np.sqrt(np.mean(differences ** 2, axis=2))


def median_pairwise_distance(distance_matrix):
    """Exclude diagonal entries and count each pair once."""
    indices = np.triu_indices(len(distance_matrix), k=1)
    values = distance_matrix[indices]
    return float(np.median(values)) if values.size else np.nan


def format_coordinates(values):
    return "-".join(f"{value:.6f}" for value in values)


region_rows = []
coordinate_rows = []
sampling_rows = []
region_diagnostics = {}

for function_id in range(1, 9):
    X = updated_data[function_id]["X"]
    y = updated_data[function_id]["y"]

    n_observations, dimensions = X.shape
    initial_count = len(initial_data[function_id]["y"])

    assert n_observations == initial_count + completed_rounds

    best_index = int(np.argmax(y))
    best_point = X[best_index]
    best_output = float(y[best_index])

    # Stable ranking makes the selection reproducible when outputs tie.
    ranked_indices = np.argsort(-y, kind="stable")
    top_count = max(3, int(np.ceil(0.25 * n_observations)))
    top_indices = ranked_indices[:top_count]

    top_X = X[top_indices]
    top_centroid = np.mean(top_X, axis=0)

    distances = rms_distance_matrix(X)
    top_distances = distances[np.ix_(top_indices, top_indices)]

    all_median_distance = median_pairwise_distance(distances)
    top_median_distance = median_pairwise_distance(top_distances)

    concentration_ratio = (
        top_median_distance / all_median_distance
        if all_median_distance > 0
        else np.nan
    )

    distances_to_best = distances[best_index]
    other_observations = np.arange(n_observations) != best_index

    neighbourhood_masks = {
        radius: (
            (distances_to_best <= radius)
            & other_observations
        )
        for radius in (0.05, 0.10, 0.15)
    }

    neighbours = neighbourhood_masks[0.10]
    neighbour_outputs = y[neighbours]

    # The latest query is the final row; compare only with earlier points.
    latest_distances = distances[-1, :-1]
    nearest_previous_index = int(np.argmin(latest_distances))

    latest_rms_distance = float(
        latest_distances[nearest_previous_index]
    )
    latest_euclidean_distance = (
        latest_rms_distance * np.sqrt(dimensions)
    )

    # Identify ties at the cutoff rather than silently treating the
    # highest-ranked subset as uniquely defined.
    cutoff_output = float(y[top_indices[-1]])
    cutoff_tie = bool(
        top_count < n_observations
        and y[ranked_indices[top_count]] == cutoff_output
    )

    region_rows.append({
        "Function": function_id,
        "Observations": n_observations,
        "Top-ranked count": top_count,
        "All median pair distance": all_median_distance,
        "Top median pair distance": top_median_distance,
        "Top / all distance ratio": concentration_ratio,
        "Other neighbours within 0.05": int(
            neighbourhood_masks[0.05].sum()
        ),
        "Other neighbours within 0.10": int(neighbours.sum()),
        "Other neighbours within 0.15": int(
            neighbourhood_masks[0.15].sum()
        ),
        "Neighbour median output": (
            float(np.median(neighbour_outputs))
            if neighbour_outputs.size else np.nan
        ),
        "Best observed output": best_output,
        "Tie at top-subset cutoff": cutoff_tie,
    })

    coordinate_rows.append({
        "Function": function_id,
        "Best observed coordinates": format_coordinates(best_point),
        "Top-ranked centroid": format_coordinates(top_centroid),
    })

    sampling_rows.append({
        "Function": function_id,
        "Latest query nearest distance (Euclidean)": (
            latest_euclidean_distance
        ),
        "Latest query nearest distance (RMS)": latest_rms_distance,
        "Latest output": float(y[-1]),
        "Nearest earlier output": float(y[nearest_previous_index]),
    })

    # Retain diagnostics for later plots and candidate interpretation.
    region_diagnostics[function_id] = {
        "best_index": best_index,
        "top_indices": top_indices,
        "top_centroid": top_centroid,
        "distance_matrix_rms": distances,
        "neighbourhood_masks": neighbourhood_masks,
        "latest_nearest_previous_index": nearest_previous_index,
    }

region_summary_df = (
    pd.DataFrame(region_rows).set_index("Function")
)
region_coordinates_df = (
    pd.DataFrame(coordinate_rows).set_index("Function")
)
sampling_summary_df = (
    pd.DataFrame(sampling_rows).set_index("Function")
)

print("Promising-region diagnostics — RMS distances")
display(
    region_summary_df.style.format({
        "All median pair distance": "{:.4f}",
        "Top median pair distance": "{:.4f}",
        "Top / all distance ratio": "{:.3f}",
        "Neighbour median output": "{:.6e}",
        "Best observed output": "{:.6e}",
    }, na_rep="—")
)

print("\nBest observed points and descriptive centroids")
with pd.option_context("display.max_colwidth", None):
    display(region_coordinates_df)

print("\nLatest-query proximity to earlier observations")
display(
    sampling_summary_df.style.format({
        "Latest query nearest distance (Euclidean)": "{:.6f}",
        "Latest query nearest distance (RMS)": "{:.6f}",
        "Latest output": "{:.6e}",
        "Nearest earlier output": "{:.6e}",
    })
)

print(
    "\nA top/all distance ratio below 1 indicates a smaller median "
    "pairwise distance in the highest-ranked subset."
)
print(
    "This is descriptive evidence of concentration, not proof of "
    "a natural cluster or a statistically significant difference."
)
print(
    "Exact output ranks, especially for near-zero values in Function 1, "
    "may not represent practically meaningful performance differences."
)

Promising-region diagnostics — RMS distances


,Observations,Top-ranked count,All median pair distance,Top median pair distance,Top / all distance ratio,Other neighbours within 0.05,Other neighbours within 0.10,Other neighbours within 0.15,Neighbour median output,Best observed output,Tie at top-subset cutoff
Function,,,,,,,,,,,
1,20,5,0.3918,0.2870,0.732,0,2,2,-1.803031e-03,7.710875e-16,False
2,20,5,0.4353,0.0183,0.042,4,5,5,5.476083e-01,6.582395e-01,False
3,25,7,0.4291,0.3668,0.855,0,2,2,-5.152745e-02,-3.096391e-02,False
4,40,10,0.3224,0.0430,0.134,7,9,10,1.219422e-01,6.152729e-01,False
5,30,8,0.4398,0.1999,0.454,4,4,4,7.950821e+03,8.515379e+03,False
6,30,8,0.3791,0.1277,0.337,3,3,5,-3.355773e-01,-2.424310e-01,False
7,40,10,0.4023,0.1863,0.463,1,4,5,2.134582e+00,2.436323e+00,False
8,50,13,0.4116,0.2641,0.642,0,4,5,9.915954e+00,9.958227e+00,False



Best observed points and descriptive centroids


,Best observed coordinates,Top-ranked centroid
Function,,
1,0.731024-0.733000,0.608489-0.678478
2,0.720690-0.003722,0.702633-0.186531
3,0.999318-0.964676-0.505507,0.520727-0.687552-0.362395
4,0.424455-0.424454-0.381378-0.414254,0.408087-0.415180-0.380988-0.425761
5,0.997181-0.999005-0.999356-0.996804,0.798121-0.972695-0.984556-0.987198
6,0.432285-0.374372-0.559617-0.651650-0.005539,0.474726-0.350823-0.551373-0.727509-0.006413
7,0.021409-0.225288-0.479072-0.145922-0.286321-0.697760,0.138061-0.271345-0.559383-0.178986-0.306093-0.731987
8,0.071169-0.178617-0.100685-0.109637-0.996594-0.554316-0.240640-0.908329,0.141703-0.182885-0.125772-0.188132-0.848582-0.557757-0.205919-0.643268



Latest-query proximity to earlier observations


,Latest query nearest distance (Euclidean),Latest query nearest distance (RMS),Latest output,Nearest earlier output
Function,,,,
1,0.326971,0.231203,-5.418764e-183,2.535001e-40
2,0.008511,0.006018,4.565338e-01,6.582395e-01
3,0.161226,0.093084,-5.532620e-02,-1.106209e-01
4,0.014990,0.007495,5.801313e-01,4.098217e-01
5,0.000235,0.000118,8.515379e+03,8.510289e+03
6,0.017641,0.007889,-3.355773e-01,-3.038324e-01
7,0.113450,0.046316,2.436323e+00,2.374721e+00
8,0.185514,0.065589,9.902243e+00,9.958227e+00



A top/all distance ratio below 1 indicates a smaller median pairwise distance in the highest-ranked subset.
This is descriptive evidence of concentration, not proof of a natural cluster or a statistically significant difference.
Exact output ranks, especially for near-zero values in Function 1, may not represent practically meaningful performance differences.


### Interpretation of the regional diagnostics

Functions 2 and 4 show the strongest concentration of highly ranked
observations, with top/all median-distance ratios of 0.042 and 0.134.
However, concentration does not imply consistently high performance.
For Function 2, the latest query was only 0.008511 Euclidean units from
the best observed point but returned approximately 0.457 rather than
0.658. Function 4 also shows substantial output variation within its
sampled neighbourhood.

Function 5's highly ranked observations favour high coordinate values,
particularly in the last three dimensions. Its latest query was only
0.000235 Euclidean units from an earlier observation and improved the
best output by approximately 0.06%. This motivates comparing less
redundant candidates before committing another evaluation to that area.

Function 6's best point and top-ranked centroid both have a fifth
coordinate close to zero. Function 7 improved at a point approximately
0.113450 Euclidean units from its nearest previous observation, providing
additional evidence for its promising region without an almost identical
repeat.

Functions 1 and 3 provide weaker evidence of a compact high-performing
region. Function 1 also requires caution because ranking extremely small
outputs can exaggerate practically negligible differences. Function 8
shows some concentration, but its highly ranked observations remain
spread across an eight-dimensional space.

These findings describe the sampled data. Adaptive query selection
contributes to the observed concentrations, and the results do not
establish natural clusters, causal variable effects or global optima.

## 6. Refit the Gaussian Process models

A separate Gaussian Process is fitted to each function using all
observations through round 10.

The Matérn smoothness settings and parameter bounds are retained from
CP_21. Continuous kernel parameters are re-estimated using the updated
data. This step does not perform a new cross-validation search.

Fitting warnings are recorded for inspection. A parameter reaching a bound
may indicate a constrained fit, but it does not by itself demonstrate
that changing the bound would improve prediction or optimisation.

Regional diagnostics will inform the subsequent comparison of candidate
locations and their proximity to existing observations.

In [7]:
# Refit the eight Gaussian Processes using observations through round 10.

import time
import warnings

import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel,
)

required_variables = ["updated_data", "completed_rounds"]

missing_variables = [
    name for name in required_variables if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Run the data-loading cell first. "
        f"Missing variables: {', '.join(missing_variables)}"
    )

gp_configurations = {
    1: {"name": "Matern_1.5_original", "nu": 1.5, "upper": 2.0},
    2: {"name": "Matern_1.5_wider", "nu": 1.5, "upper": 10.0},
    3: {"name": "Matern_1.5_original", "nu": 1.5, "upper": 2.0},
    4: {"name": "Matern_1.5_wider", "nu": 1.5, "upper": 10.0},
    5: {"name": "Matern_1.5_original", "nu": 1.5, "upper": 2.0},
    6: {"name": "Matern_2.5_original", "nu": 2.5, "upper": 2.0},
    7: {"name": "Matern_2.5_original", "nu": 2.5, "upper": 2.0},
    8: {"name": "Matern_2.5_wider", "nu": 2.5, "upper": 10.0},
}

gp_models = {}
gp_fit_rows = []
gp_warning_rows = []

for function_id in range(1, 9):
    X = updated_data[function_id]["X"]
    y = updated_data[function_id]["y"]

    configuration = gp_configurations[function_id]
    dimensions = X.shape[1]

    kernel = (
        ConstantKernel(
            constant_value=1.0,
            constant_value_bounds=(1e-3, 1e3),
        )
        * Matern(
            length_scale=np.full(dimensions, 0.2),
            length_scale_bounds=(0.02, configuration["upper"]),
            nu=configuration["nu"],
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1),
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        alpha=1e-10,
        normalize_y=True,
        n_restarts_optimizer=3,
        random_state=42,
    )

    start_time = time.perf_counter()

    with warnings.catch_warnings(record=True) as recorded_warnings:
        warnings.simplefilter("always")
        model.fit(X, y)

    elapsed_seconds = time.perf_counter() - start_time

    gp_models[function_id] = model

    gp_fit_rows.append({
        "Function": function_id,
        "Observations": len(y),
        "GP configuration": configuration["name"],
        "Fitted kernel": str(model.kernel_),
        "Log-marginal likelihood": (
            model.log_marginal_likelihood_value_
        ),
        "GP fit seconds": elapsed_seconds,
    })

    for warning in recorded_warnings:
        gp_warning_rows.append({
            "Function": function_id,
            "Category": warning.category.__name__,
            "Message": str(warning.message),
        })

gp_fit_summary_df = (
    pd.DataFrame(gp_fit_rows)
    .set_index("Function")
    .sort_index()
)

gp_warnings_df = pd.DataFrame(
    gp_warning_rows,
    columns=["Function", "Category", "Message"],
)

display(
    gp_fit_summary_df.style.format({
        "Log-marginal likelihood": "{:.6f}",
        "GP fit seconds": "{:.3f}",
    })
)

print(
    f"\nAll eight Gaussian Processes were fitted using observations "
    f"through round {completed_rounds}."
)
print(
    "Kernel configurations were retained from CP_21; "
    "continuous kernel parameters were re-estimated."
)
print("\nRecorded fitting warnings:", len(gp_warnings_df))

if not gp_warnings_df.empty:
    with pd.option_context("display.max_colwidth", None):
        display(gp_warnings_df)

,Observations,GP configuration,Fitted kernel,Log-marginal likelihood,GP fit seconds
Function,,,,,
1,20,Matern_1.5_original,"1.07**2 * Matern(length_scale=[2, 0.039], nu=1.5) + WhiteKernel(noise_level=1.25e-08)",-26.678973,0.445
2,20,Matern_1.5_wider,"1.08**2 * Matern(length_scale=[0.11, 10], nu=1.5) + WhiteKernel(noise_level=0.0762)",-17.623983,0.131
3,25,Matern_1.5_original,"1.23**2 * Matern(length_scale=[0.51, 2, 0.157], nu=1.5) + WhiteKernel(noise_level=0.00362)",-30.016038,0.220
4,40,Matern_1.5_wider,"4.77**2 * Matern(length_scale=[3.91, 3.62, 3.7, 3.8], nu=1.5) + WhiteKernel(noise_level=0.000341)",4.570818,0.265
5,30,Matern_1.5_original,"1.36**2 * Matern(length_scale=[1.82, 2, 1.31, 2], nu=1.5) + WhiteKernel(noise_level=0.00831)",-5.776324,0.163
6,30,Matern_2.5_original,"1.66**2 * Matern(length_scale=[1, 1.32, 1.39, 1.21, 2], nu=2.5) + WhiteKernel(noise_level=0.0253)",-20.211213,0.256
7,40,Matern_2.5_original,"0.634**2 * Matern(length_scale=[0.731, 1.78, 2, 0.183, 0.207, 0.215], nu=2.5) + WhiteKernel(noise_level=1.89e-10)",-25.373387,0.451
8,50,Matern_2.5_wider,"3.12**2 * Matern(length_scale=[3.3, 5.26, 2.74, 4.48, 10, 5.08, 3.44, 10], nu=2.5) + WhiteKernel(noise_level=1e-10)",18.294021,0.601



All eight Gaussian Processes were fitted using observations through round 10.
Kernel configurations were retained from CP_21; continuous kernel parameters were re-estimated.

Recorded fitting warnings: 10


,Function,Category,Message
0,1,ConvergenceWarning,The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
1,2,ConvergenceWarning,The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
2,3,ConvergenceWarning,The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
3,5,ConvergenceWarning,The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
4,5,ConvergenceWarning,The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
5,6,ConvergenceWarning,The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
6,7,ConvergenceWarning,The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
7,8,ConvergenceWarning,The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
8,8,ConvergenceWarning,The optimal value found for dimension 7 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
9,8,ConvergenceWarning,The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-10. Decreasing the bound and calling fit again may find a better value.


## 7. Refit the auxiliary models

The auxiliary models from CP_21 are retained for comparison:

- Function 5: a neural network with one hidden layer of eight neurons.
- Function 7: linear regression.

Both models are fitted using all observations through round 10. Input
and target scaling are fitted within each model pipeline, and predictions
are returned in the original output units.

In round 10, the neural network's absolute error for Function 5 was
approximately 461, compared with 470 for the GP. This small difference
does not establish a consistent advantage. For Function 7, linear
regression had a substantially larger error than the GP, approximately
1.344 versus 0.156.

The auxiliary predictions therefore provide a diagnostic comparison.
Agreement between models does not guarantee accuracy, and disagreement
does not automatically identify which prediction is correct.

In [8]:
# Refit auxiliary models using observations through round 10.
# Predictions are automatically returned in original output units.

import time
import warnings

import pandas as pd

from IPython.display import display
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

required_variables = ["updated_data", "completed_rounds"]

missing_variables = [
    name for name in required_variables if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Run the data-loading cell first. "
        f"Missing variables: {', '.join(missing_variables)}"
    )

auxiliary_estimators = {
    5: {
        "name": "NN (8,)",
        "estimator": MLPRegressor(
            hidden_layer_sizes=(8,),
            activation="tanh",
            solver="lbfgs",
            alpha=0.1,
            max_iter=2000,
            random_state=42,
        ),
    },
    7: {
        "name": "Linear regression",
        "estimator": LinearRegression(),
    },
}

auxiliary_models = {}
auxiliary_fit_rows = []
auxiliary_warning_rows = []

for function_id, specification in auxiliary_estimators.items():
    X = updated_data[function_id]["X"]
    y = updated_data[function_id]["y"]

    input_pipeline = Pipeline([
        ("input_scaler", StandardScaler()),
        ("regressor", specification["estimator"]),
    ])

    model = TransformedTargetRegressor(
        regressor=input_pipeline,
        transformer=StandardScaler(),
    )

    start_time = time.perf_counter()

    with warnings.catch_warnings(record=True) as recorded_warnings:
        warnings.simplefilter("always")
        model.fit(X, y)

    elapsed_seconds = time.perf_counter() - start_time

    auxiliary_models[function_id] = model

    auxiliary_fit_rows.append({
        "Function": function_id,
        "Observations": len(y),
        "Auxiliary model": specification["name"],
        "Fit seconds": elapsed_seconds,
    })

    for warning in recorded_warnings:
        auxiliary_warning_rows.append({
            "Function": function_id,
            "Category": warning.category.__name__,
            "Message": str(warning.message),
        })

auxiliary_fit_summary_df = (
    pd.DataFrame(auxiliary_fit_rows)
    .set_index("Function")
    .sort_index()
)

auxiliary_warnings_df = pd.DataFrame(
    auxiliary_warning_rows,
    columns=["Function", "Category", "Message"],
)

display(
    auxiliary_fit_summary_df.style.format({
        "Fit seconds": "{:.3f}",
    })
)

print(
    "\nAuxiliary models fitted for Functions 5 and 7 using "
    f"observations through round {completed_rounds}."
)
print("Predictions will be returned in the original output units.")

if auxiliary_warnings_df.empty:
    print("No auxiliary fitting warnings were recorded.")
else:
    print(
        "Recorded auxiliary fitting warnings:",
        len(auxiliary_warnings_df),
    )
    with pd.option_context("display.max_colwidth", None):
        display(auxiliary_warnings_df)

,Observations,Auxiliary model,Fit seconds
Function,,,
5,30,"NN (8,)",0.096
7,40,Linear regression,0.020



Auxiliary models fitted for Functions 5 and 7 using observations through round 10.
Predictions will be returned in the original output units.
No auxiliary fitting warnings were recorded.


## 8. Generate and compare candidate queries

For each function, candidate inputs are generated through:

- A global search with 100,000 candidates.
- A local search with 10,000 candidates within a coordinate-wise
  half-width of 0.10 around the best observed point.
- A narrower local search with 10,000 candidates using a half-width of 0.03.

Candidates are rounded to six decimal places before evaluation.
Duplicates within each search and points within 0.000001 Euclidean units
of existing observations are excluded.

Candidates are ranked by UCB = GP mean + 2 × GP predictive standard
deviation. The reported uncertainty includes the fitted white-noise
component and should not be interpreted as purely reducible uncertainty.

For Function 5, a second proposal from each search excludes candidates
within 0.01 Euclidean units of any existing observation. This exploratory
threshold allows comparison with less repetitive queries after the small
gain from the near-repeat in round 10. It is not a tuned optimum and may
exclude useful local refinements.

Each proposal also records its RMS distance to the best observed point
and to the top-ranked centroid, plus the number of observations within
an RMS radius of 0.10. These quantities describe its sampled neighbourhood;
they do not prove cluster membership or predict improvement.

This cell prepares proposals only. Final selection follows inspection
of their scores, distances and auxiliary predictions.

In [9]:
# Generate global and local candidates and compare their UCB scores.
# Include a distance-filtered alternative for Function 5.

import time

import numpy as np
import pandas as pd

from IPython.display import display
from scipy.spatial import cKDTree

required_variables = [
    "updated_data",
    "gp_models",
    "auxiliary_models",
    "region_diagnostics",
]

missing_variables = [
    name for name in required_variables if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Run the preceding data, diagnostic and model-fitting cells first. "
        f"Missing variables: {', '.join(missing_variables)}"
    )

GLOBAL_CANDIDATES = 100_000
LOCAL_CANDIDATES = 10_000
PREDICTION_BATCH_SIZE = 5_000
UCB_WEIGHT = 2.0
DUPLICATE_TOLERANCE = 1e-6
F5_ALTERNATIVE_DISTANCE = 0.01
NEIGHBOURHOOD_RMS_RADIUS = 0.10
BASE_SEED = 42


def predict_in_batches(model, X, batch_size):
    """Return predictive means and standard deviations in original units."""
    mean_parts = []
    std_parts = []

    for start in range(0, len(X), batch_size):
        mean, std = model.predict(
            X[start:start + batch_size],
            return_std=True,
        )
        mean_parts.append(mean)
        std_parts.append(std)

    return np.concatenate(mean_parts), np.concatenate(std_parts)


def prepare_candidates(raw_candidates, observation_tree):
    """Round, enforce submission bounds, remove duplicates and repeats."""
    candidates = np.unique(
        np.round(raw_candidates, decimals=6),
        axis=0,
    )

    # Submission coordinates must start with 0, so exclude rounded 1.000000.
    valid_bounds = (
        (candidates >= 0.0) & (candidates < 1.0)
    ).all(axis=1)
    candidates = candidates[valid_bounds]

    if len(candidates) == 0:
        raise RuntimeError("No candidates remain after checking bounds.")

    nearest_distances, _ = observation_tree.query(candidates, k=1)
    keep = nearest_distances > DUPLICATE_TOLERANCE

    candidates = candidates[keep]
    nearest_distances = nearest_distances[keep]

    if len(candidates) == 0:
        raise RuntimeError("No candidates remain after removing repeats.")

    return candidates, nearest_distances


candidate_rows = []
search_time_rows = []
candidate_search_settings = {
    "global_candidates": GLOBAL_CANDIDATES,
    "local_candidates": LOCAL_CANDIDATES,
    "local_half_widths": [0.10, 0.03],
    "ucb_weight": UCB_WEIGHT,
    "duplicate_tolerance_euclidean": DUPLICATE_TOLERANCE,
    "f5_alternative_distance_euclidean": F5_ALTERNATIVE_DISTANCE,
    "neighbourhood_radius_rms": NEIGHBOURHOOD_RMS_RADIUS,
    "base_seed": BASE_SEED,
}

for function_id in range(1, 9):
    start_time = time.perf_counter()

    X = updated_data[function_id]["X"]
    y = updated_data[function_id]["y"]
    dimensions = X.shape[1]

    best_point = X[int(np.argmax(y))]
    current_best = float(np.max(y))
    top_centroid = region_diagnostics[function_id]["top_centroid"]

    observation_tree = cKDTree(X)
    rng = np.random.default_rng(BASE_SEED + function_id)

    search_specs = [
        ("Global", None, GLOBAL_CANDIDATES),
        ("Local 0.10", 0.10, LOCAL_CANDIDATES),
        ("Local 0.03", 0.03, LOCAL_CANDIDATES),
    ]

    for search_name, half_width, count in search_specs:
        if half_width is None:
            raw_candidates = rng.uniform(
                0.0, 1.0, size=(count, dimensions)
            )
        else:
            lower = np.maximum(0.0, best_point - half_width)
            upper = np.minimum(1.0, best_point + half_width)

            raw_candidates = rng.uniform(
                lower,
                upper,
                size=(count, dimensions),
            )

        candidates, nearest_distances = prepare_candidates(
            raw_candidates,
            observation_tree,
        )

        means, stds = predict_in_batches(
            gp_models[function_id],
            candidates,
            PREDICTION_BATCH_SIZE,
        )
        ucb_scores = means + UCB_WEIGHT * stds

        if not (
            np.isfinite(means).all()
            and np.isfinite(stds).all()
            and np.isfinite(ucb_scores).all()
        ):
            raise RuntimeError(
                f"Function {function_id}: non-finite predictions."
            )

        proposal_filters = [
            ("Standard", np.ones(len(candidates), dtype=bool))
        ]

        if function_id == 5:
            proposal_filters.append((
                "Distance >= 0.01",
                nearest_distances >= F5_ALTERNATIVE_DISTANCE,
            ))

        for proposal_name, eligible_mask in proposal_filters:
            eligible_indices = np.flatnonzero(eligible_mask)

            if len(eligible_indices) == 0:
                raise RuntimeError(
                    f"Function {function_id}, {search_name}: "
                    f"no candidates satisfy '{proposal_name}'."
                )

            selected_index = eligible_indices[
                np.argmax(ucb_scores[eligible_indices])
            ]

            point = candidates[selected_index]

            distances_rms = np.sqrt(
                np.mean((X - point) ** 2, axis=1)
            )

            auxiliary_prediction = np.nan
            if function_id in auxiliary_models:
                auxiliary_prediction = float(
                    auxiliary_models[function_id].predict(
                        point.reshape(1, -1)
                    )[0]
                )

            candidate_rows.append({
                "Function": function_id,
                "Search": search_name,
                "Proposal": proposal_name,
                "Query": "-".join(
                    f"{coordinate:.6f}" for coordinate in point
                ),
                "Current best": current_best,
                "GP mean": float(means[selected_index]),
                "GP std": float(stds[selected_index]),
                "UCB": float(ucb_scores[selected_index]),
                "Nearest observation distance": float(
                    nearest_distances[selected_index]
                ),
                "Distance to best (RMS)": float(
                    np.sqrt(np.mean((point - best_point) ** 2))
                ),
                "Distance to top centroid (RMS)": float(
                    np.sqrt(np.mean((point - top_centroid) ** 2))
                ),
                "Neighbours within RMS 0.10": int(
                    np.sum(
                        distances_rms <= NEIGHBOURHOOD_RMS_RADIUS
                    )
                ),
                "Auxiliary prediction": auxiliary_prediction,
                "Eligible candidates": len(eligible_indices),
            })

    search_time_rows.append({
        "Function": function_id,
        "Candidate search seconds": (
            time.perf_counter() - start_time
        ),
    })

candidate_comparison_df = pd.DataFrame(candidate_rows)

candidate_search_times_df = (
    pd.DataFrame(search_time_rows).set_index("Function")
)

score_columns = [
    "Function",
    "Search",
    "Proposal",
    "Query",
    "Current best",
    "GP mean",
    "GP std",
    "UCB",
    "Nearest observation distance",
    "Auxiliary prediction",
]

display(
    candidate_comparison_df[score_columns].style.format({
        "Current best": "{:.6e}",
        "GP mean": "{:.6e}",
        "GP std": "{:.6e}",
        "UCB": "{:.6e}",
        "Nearest observation distance": "{:.6f}",
        "Auxiliary prediction": "{:.6e}",
    }, na_rep="—")
)

print("\nRegional context of each proposal")
display(
    candidate_comparison_df[[
        "Function",
        "Search",
        "Proposal",
        "Distance to best (RMS)",
        "Distance to top centroid (RMS)",
        "Neighbours within RMS 0.10",
        "Eligible candidates",
    ]].style.format({
        "Distance to best (RMS)": "{:.6f}",
        "Distance to top centroid (RMS)": "{:.6f}",
    })
)

print("\nCandidate search time")
display(
    candidate_search_times_df.style.format({
        "Candidate search seconds": "{:.3f}",
    })
)

print(
    "\nProposals are ready for comparison; final queries "
    "have not yet been selected."
)
print(
    "F5 alternatives use the same candidate pools and predictions "
    "as the standard proposals."
)

,Function,Search,Proposal,Query,Current best,GP mean,GP std,UCB,Nearest observation distance,Auxiliary prediction
0,1,Global,Standard,0.997786-0.810962,7.710875e-16,-1.043251e-04,7.300194e-04,1.355714e-03,0.189806,—
1,1,Local 0.10,Standard,0.830609-0.812268,7.710875e-16,-1.052069e-04,7.252653e-04,1.345324e-03,0.127282,—
2,1,Local 0.03,Standard,0.761008-0.749102,7.710875e-16,1.710411e-04,2.942023e-04,7.594458e-04,0.034034,—
3,2,Global,Standard,0.690418-0.006928,6.582395e-01,5.722578e-01,7.806085e-02,7.283795e-01,0.007045,—
4,2,Local 0.10,Standard,0.690168-0.000257,6.582395e-01,5.721264e-01,7.818321e-02,7.284928e-01,0.001670,—
5,2,Local 0.03,Standard,0.690695-0.000239,6.582395e-01,5.724827e-01,7.799206e-02,7.284668e-01,0.001147,—
6,3,Global,Standard,0.989378-0.010523-0.715547,-3.096391e-02,-8.934356e-02,6.852852e-02,4.771347e-02,0.458327,—
7,3,Local 0.10,Standard,0.999881-0.871291-0.445814,-3.096391e-02,-3.787429e-02,2.682529e-02,1.577629e-02,0.110835,—
8,3,Local 0.03,Standard,0.998783-0.941843-0.475607,-3.096391e-02,-3.154443e-02,1.848866e-02,5.432891e-03,0.037625,—
9,4,Global,Standard,0.378762-0.416287-0.401547-0.396799,6.152729e-01,1.625750e-01,2.513929e-01,6.653608e-01,0.042546,—



Regional context of each proposal


,Function,Search,Proposal,Distance to best (RMS),Distance to top centroid (RMS),Neighbours within RMS 0.10,Eligible candidates
0,1,Global,Standard,0.196520,0.290779,0,99996
1,1,Local 0.10,Standard,0.090002,0.183354,1,10000
2,1,Local 0.03,Standard,0.024066,0.118848,3,10000
3,2,Global,Standard,0.021525,0.127292,6,99998
4,2,Local 0.10,Standard,0.021721,0.132010,6,9999
5,2,Local 0.03,Standard,0.021352,0.131998,6,9998
6,3,Global,Standard,0.564099,0.517274,0,99999
7,3,Local 0.10,Standard,0.063990,0.300170,2,10000
8,3,Local 0.03,Standard,0.021723,0.319384,3,10000
9,4,Global,Standard,0.026768,0.023036,10,100000



Candidate search time


,Candidate search seconds
Function,
1,0.595
2,0.386
3,0.445
4,0.468
5,0.449
6,0.547
7,0.635
8,0.689



Proposals are ready for comparison; final queries have not yet been selected.
F5 alternatives use the same candidate pools and predictions as the standard proposals.


## 9. Select and validate the round-11 queries

For Functions 1–4 and 6–8, I select the highest-UCB standard proposal
using the full numerical scores.

For Function 5, I select the highest-UCB proposal satisfying a minimum
Euclidean distance of 0.01 from every existing observation. The standard
winner is only approximately 0.000018 from an existing point. The
distance-filtered alternative increases this separation to approximately
0.013717 while reducing UCB by approximately 0.53%.

This is an explicit adjustment to the previous selection policy, motivated
by the small gain from the near-repeat in round 10. Greater separation
does not guarantee greater information gain or improved performance.

The selected queries target different parts of the sampled space:

- Function 1: a location without observations inside the diagnostic
  RMS radius of 0.10, reflecting uncertainty-driven exploration.
- Function 2: the repeatedly sampled region with the second coordinate
  near zero. Its proximity to existing observations remains a limitation.
- Function 3: a sparsely sampled location away from the current best,
  selected for its uncertainty contribution.
- Function 4: the concentrated interior region near the top-ranked
  centroid, where observed outputs nevertheless vary substantially.
- Function 5: a less repetitive point within the high-coordinate region.
- Function 6: local refinement with the fifth coordinate near zero.
- Function 7: local refinement around the newly improved region.
- Function 8: a location with no observations inside the diagnostic
  RMS radius, prioritising uncertainty over the higher local means.

These descriptions refer to neighbourhoods in the observed data, not
formally identified natural clusters.

Validation checks dimensions, coordinate format, bounds and separation
from existing observations. It does not guarantee improvement. Only the
coordinate strings are submitted to the portal; predictions and UCB
scores are retained for subsequent evaluation.

In [10]:
# Select, validate and display the eight round-11 queries.
# F5 uses the documented distance-filtered selection policy.

import re

import numpy as np
import pandas as pd
from IPython.display import display

required_variables = [
    "candidate_comparison_df",
    "updated_data",
    "submission_round",
    "DUPLICATE_TOLERANCE",
    "F5_ALTERNATIVE_DISTANCE",
]

missing_variables = [
    name for name in required_variables if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Run the candidate-generation cell first. "
        f"Missing variables: {', '.join(missing_variables)}"
    )

assert submission_round == 11, (
    "This selection cell is intended for round 11."
)

selected_rows = []

for function_id in range(1, 9):
    proposal_type = (
        "Distance >= 0.01" if function_id == 5 else "Standard"
    )

    eligible = candidate_comparison_df.loc[
        (candidate_comparison_df["Function"] == function_id)
        & (candidate_comparison_df["Proposal"] == proposal_type)
    ]

    if eligible.empty:
        raise RuntimeError(
            f"Function {function_id}: no eligible proposals."
        )

    assert np.isfinite(eligible["UCB"].to_numpy()).all()

    winner = eligible.loc[eligible["UCB"].idxmax()].copy()
    query = str(winner["Query"])

    X = updated_data[function_id]["X"]
    dimensions = X.shape[1]

    expected_pattern = (
        rf"0\.\d{{6}}(?:-0\.\d{{6}}){{{dimensions - 1}}}"
    )

    assert re.fullmatch(expected_pattern, query), (
        f"Function {function_id}: invalid submission format."
    )

    coordinates = np.array(
        [float(value) for value in query.split("-")],
        dtype=float,
    )

    assert coordinates.shape == (dimensions,)
    assert np.isfinite(coordinates).all()
    assert ((coordinates >= 0.0) & (coordinates < 1.0)).all()

    nearest_distance = float(
        np.min(np.linalg.norm(X - coordinates, axis=1))
    )

    assert nearest_distance > DUPLICATE_TOLERANCE, (
        f"Function {function_id}: query is too close to an "
        "existing observation under the duplicate tolerance."
    )

    if function_id == 5:
        assert nearest_distance >= F5_ALTERNATIVE_DISTANCE, (
            "Function 5: query fails the additional distance constraint."
        )

    winner["Nearest observation distance"] = nearest_distance
    winner["Selection rule"] = (
        "Maximum UCB with minimum distance 0.01"
        if function_id == 5
        else "Maximum standard UCB"
    )

    selected_rows.append(winner.to_dict())

selected_queries_df = (
    pd.DataFrame(selected_rows)
    .set_index("Function")
    .sort_index()
    .rename(columns={
        "Search": "Selected search",
        "GP mean": "Predicted mean",
        "GP std": "Predicted std",
    })
)

assert selected_queries_df.index.tolist() == list(range(1, 9))

# Retain the F5 comparison for documentation.
f5_standard = candidate_comparison_df.loc[
    (candidate_comparison_df["Function"] == 5)
    & (candidate_comparison_df["Proposal"] == "Standard")
]

f5_standard_best = f5_standard.loc[f5_standard["UCB"].idxmax()]
f5_selected_ucb = float(selected_queries_df.loc[5, "UCB"])

f5_ucb_reduction = (
    float(f5_standard_best["UCB"]) - f5_selected_ucb
)
f5_ucb_reduction_pct = (
    100.0 * f5_ucb_reduction / abs(float(f5_standard_best["UCB"]))
)

display_columns = [
    "Selected search",
    "Proposal",
    "Query",
    "Predicted mean",
    "Predicted std",
    "UCB",
    "Nearest observation distance",
    "Auxiliary prediction",
]

display(
    selected_queries_df[display_columns].style.format({
        "Predicted mean": "{:.6e}",
        "Predicted std": "{:.6e}",
        "UCB": "{:.6e}",
        "Nearest observation distance": "{:.6f}",
        "Auxiliary prediction": "{:.6e}",
    }, na_rep="—")
)

submission_df = (
    selected_queries_df[["Query"]]
    .rename(columns={"Query": "Query for submission"})
)

with pd.option_context("display.max_colwidth", None):
    display(submission_df)

print("\nRound 11 — queries for manual submission\n")

for function_id, row in submission_df.iterrows():
    print(f"Function {function_id}: {row['Query for submission']}")

print(
    f"\nF5 UCB reduction relative to its standard winner: "
    f"{f5_ucb_reduction:.6f} ({f5_ucb_reduction_pct:.3f}%)."
)
print("\nAll eight queries passed validation.")
print(
    "Validation confirms format, bounds and the configured "
    "distance constraints; it does not guarantee improvement."
)
print(
    "Queries are prepared; this code does not submit them to the portal."
)

,Selected search,Proposal,Query,Predicted mean,Predicted std,UCB,Nearest observation distance,Auxiliary prediction
Function,,,,,,,,
1,Global,Standard,0.997786-0.810962,-1.043251e-04,7.300194e-04,1.355714e-03,0.189806,—
2,Local 0.10,Standard,0.690168-0.000257,5.721264e-01,7.818321e-02,7.284928e-01,0.001670,—
3,Global,Standard,0.989378-0.010523-0.715547,-8.934356e-02,6.852852e-02,4.771347e-02,0.458327,—
4,Local 0.03,Standard,0.403515-0.409981-0.384203-0.416426,4.052177e-01,2.003610e-01,8.059397e-01,0.006476,—
5,Local 0.03,Distance >= 0.01,0.998659-0.983879-0.999338-0.999448,8.135976e+03,3.014681e+02,8.738912e+03,0.013717,8.102115e+03
6,Local 0.03,Standard,0.455442-0.366608-0.556940-0.676688-0.001142,-3.607229e-01,1.027307e-01,-1.552615e-01,0.023198,—
7,Local 0.03,Standard,0.003443-0.196129-0.472527-0.128049-0.314261-0.673736,2.388085e+00,9.284172e-02,2.573769e+00,0.053788,1.206557e+00
8,Global,Standard,0.009105-0.098331-0.085665-0.149083-0.983758-0.543950-0.329628-0.121389,9.902413e+00,4.844239e-02,9.999297e+00,0.380851,—


,Query for submission
Function,
1,0.997786-0.810962
2,0.690168-0.000257
3,0.989378-0.010523-0.715547
4,0.403515-0.409981-0.384203-0.416426
5,0.998659-0.983879-0.999338-0.999448
6,0.455442-0.366608-0.556940-0.676688-0.001142
7,0.003443-0.196129-0.472527-0.128049-0.314261-0.673736
8,0.009105-0.098331-0.085665-0.149083-0.983758-0.543950-0.329628-0.121389



Round 11 — queries for manual submission

Function 1: 0.997786-0.810962
Function 2: 0.690168-0.000257
Function 3: 0.989378-0.010523-0.715547
Function 4: 0.403515-0.409981-0.384203-0.416426
Function 5: 0.998659-0.983879-0.999338-0.999448
Function 6: 0.455442-0.366608-0.556940-0.676688-0.001142
Function 7: 0.003443-0.196129-0.472527-0.128049-0.314261-0.673736
Function 8: 0.009105-0.098331-0.085665-0.149083-0.983758-0.543950-0.329628-0.121389

F5 UCB reduction relative to its standard winner: 46.475371 (0.529%).

All eight queries passed validation.
Validation confirms format, bounds and the configured distance constraints; it does not guarantee improvement.
Queries are prepared; this code does not submit them to the portal.


## 10. Round-11 submission summary

Eight queries have been prepared and validated using observations from
ten completed feedback rounds.

The final selection includes three global proposals (Functions 1, 3
and 8) and five local proposals (Functions 2, 4, 5, 6 and 7).

Function 5 uses an explicit minimum-distance constraint of 0.01
Euclidean units. Its selected query is approximately 0.013717 from the
nearest observation, compared with approximately 0.000018 for the
unconstrained winner. This adjustment reduces UCB by 46.475371,
or approximately 0.529%, while avoiding another almost identical query.

The notebook records the historical prediction errors, regional
diagnostics, model configurations, fitting warnings, candidate scores
and final selection rule. The regional analysis is descriptive:
no formal clustering model was fitted.

Validation confirms dimensionality, bounds, six-decimal formatting
and the applicable distance constraints. It does not establish that
the queries will improve the observed best values.

The queries must be submitted manually through the capstone portal.
Round-11 outputs remain unknown. Once feedback is available, the
recorded predictions will be compared with observed evaluations
before updating the models.